# Prep 6: Variational Autoencoder (VAE)

Source: [Official PyTorch VAE example](https://github.com/pytorch/examples/tree/main/vae)

## Learning purpose

Build a mental model of how a VAE compresses, reconstructs, and samples images.

## Why this matters

A VAE is a model that learns to compress an image into a small hidden code, then rebuild the image from that code. The hidden code lives in a **latent space**, which means a space of learned numbers that represent useful image features rather than raw pixels. This notebook makes that idea visible by saving reconstruction grids, sample grids, and possibly interpolation grids. Later, diffusion models will reuse the same broad idea: generate images by working with learned representations instead of only thinking in raw pixels.

## Core idea

For MNIST, each digit image has `28 × 28 = 784` pixel values. A basic autoencoder learns this pattern:

```text
image pixels → encoder → small code → decoder → rebuilt image
```

The PyTorch example compresses each image into a 20-number latent representation, then decodes that representation back into 784 pixel values. This bottleneck matters because the model cannot simply copy every pixel; it has to learn useful digit structure such as loops, strokes, and slants.

## What makes it variational?

A regular autoencoder can map an image to one exact code. A VAE maps an image to a small probability cloud:

```text
image → mean + spread → sampled latent code → rebuilt image
```

In the official code, the encoder returns `mu` and `logvar`. `mu` is the center of the cloud, `logvar` describes its spread, and `z` is a sampled point from that cloud. This organized sampling step is what lets the decoder create new digit-like images from random latent codes.

## Training loop in plain language

For each batch of images, the VAE:

1. reads real digit images,
2. encodes each image into `mu` and `logvar`,
3. samples a latent code `z`,
4. decodes `z` into a reconstructed image,
5. measures reconstruction loss: how close the rebuilt image is to the original,
6. measures KL loss: how well the latent clouds stay organized for sampling,
7. updates the model weights to reduce the combined loss.

```text
total loss = reconstruction loss + KL loss
```

## What the saved grids should show

- **Reconstruction grid:** original digits beside rebuilt digits. This checks whether the latent code keeps enough information to preserve digit identity.
- **Sample grid:** random latent codes decoded into images. This checks whether random points in latent space become digit-like outputs.
- **Interpolation grid:** a smooth walk between two latent codes. This checks whether nearby latent points create gradual visual changes.

Blurry but recognizable digits are a useful early result. The goal is not perfect handwriting; the goal is to make compression, sampling, and latent-space structure visible.


## Notebook stage 1: setup

This cell prepares the notebook before any model training happens. It imports the PyTorch tools used later, finds the project root by looking for `pyproject.toml`, sets a fixed random seed, chooses `cuda` when a GPU is available, and creates `outputs/prep/vae/` for generated image grids.

- **Project root**: the repository folder, used so paths do not depend on where Jupyter started.
- **Device**: the hardware target for tensor math, usually `cuda` for GPU or `cpu` otherwise.
- **Seed**: a starting value for random number generation, used to make repeated runs easier to compare.
- **Output directory**: the folder where reconstruction, sample, and interpolation images will be saved.


In [3]:
# Setup: imports, project paths, seed, and device

from pathlib import Path

import torch
from torch import nn, optim
from torch.nn import functional as F
from torchvision import datasets, transforms
from torchvision.utils import save_image


def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing pyproject.toml")


torch.manual_seed(1)

project_root = find_project_root()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
output_dir = project_root / "outputs" / "prep" / "vae"
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root: {project_root}")
print(f"Using device: {device}")
print(f"Saving VAE artifacts to: {output_dir}")

Project root: C:\Users\giloz\dev\visual-genai-lab
Using device: cuda
Saving VAE artifacts to: C:\Users\giloz\dev\visual-genai-lab\outputs\prep\vae


## Notebook stage 2: MNIST data loaders

This cell downloads MNIST and wraps it in `DataLoader` objects. MNIST is a dataset of small handwritten digit images; each image is `28 × 28` grayscale pixels. The VAE will learn from batches of these images, then we will inspect whether it can reconstruct digits and sample new ones.

- **Transform**: `transforms.ToTensor()` converts each image into a PyTorch tensor, changes pixel values from `0–255` into `0.0–1.0`, and gives each image shape `[1, 28, 28]`.
- **Pinned memory**: `pin_memory=True` can speed up CPU-to-GPU transfer, but it does not change the image data and does not move data to the GPU by itself.


In [ ]:
# MNIST data: download digits and create training/test loaders
# Source: https://github.com/pytorch/examples/tree/main/vae

batch_size = 128
data_dir = project_root / "data"

transform = transforms.ToTensor()

train_loader = torch.utils.data.DataLoader(
    datasets.MNIST(
        root=data_dir,
        train=True,
        download=True,
        transform=transform,
    ),
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

test_loader = torch.utils.data.DataLoader(
    datasets.MNIST(
        root=data_dir,
        train=False,
        download=True,
        transform=transform,
    ),
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=(device.type == "cuda"),
)

images, labels = next(iter(train_loader))

print(f"Training batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")
print(f"Image batch shape: {images.shape}")
print(f"Label batch shape: {labels.shape}")
print(f"First labels: {labels[:10].tolist()}")